### install all the libraries

In [3]:
# Remove OpenAI related packages and install HuggingFace packages
!pip install langchain faiss-cpu tiktoken huggingface-hub sentence-transformers transformers


[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
pip install langchain-community

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
pip install pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Hugging-Face

In [7]:
from dotenv import load_dotenv
import os
import faiss
import tiktoken
import pypdf

from langchain_community.document_loaders import DirectoryLoader, TextLoader, UnstructuredPDFLoader, OnlinePDFLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Replace OpenAI with HuggingFace imports
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFaceHub
from langchain.vectorstores import FAISS

from langchain.chains.question_answering import load_qa_chain
from langchain.chains import RetrievalQA

# Load environment variables
load_dotenv()
hf_token = os.getenv("HUGGINGFACE_API_KEY")

# Initialize HuggingFace
from huggingface_hub import login
login(token=hf_token)

INFO:faiss.loader:Loading faiss with AVX2 support.
INFO:faiss.loader:Successfully loaded faiss with AVX2 support.


### import own pdf file

In [8]:
loader = DirectoryLoader("data" , glob="./*.pdf" ,loader_cls=PyPDFLoader)
document = loader.load() 

## split the text 

In [9]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(document)

In [8]:
type(texts)

list

In [9]:
len(texts)

3664

In [10]:
texts[3000]

Document(metadata={'source': 'data\\software-evolution-and-maintenance.pdf', 'page': 210}, page_content='information system that significantly resists modification and evolution to meet new\nand constantly changing business requirements.” Supporting both the definitions are\na set of acceptable features of a legacy system:\nr large with millions of lines of code;\nr geriatric, often more than 10 years old;\nr written in obsolete programming languages;\nSoftwareEvolutionandMaintenance: APractitioner’sApproach , First Edition.\nPriyadarshi Tripathy and Kshirasagar Naik.\n© 2015 John Wiley & Sons, Inc. Published 2015 by John Wiley & Sons, Inc.\n187\nwww.it-ebooks.info')

In [11]:
texts[3000].metadata

{'source': 'data\\software-evolution-and-maintenance.pdf', 'page': 210}

In [12]:
texts[2800].metadata['source']

'data\\software-evolution-and-maintenance.pdf'

### HuggingFace Embedding 

In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Create HuggingFace embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_20456\472148012.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-mpnet-base-v2


### VectorDB

In [11]:
# Create vector from text (this part remains the same)
docsearch = FAISS.from_documents(texts, embeddings)

In [12]:
query1 = "Explain the three areas of the SCM (Software Configuration Management) functionalities."
# query1 = "what is seven-segment display."
# query1 = "Who is the president of the United States?"
# query1 = "List down design values in interaction designs on interactive application design."
# query1 = "Why Personas Are Effective in interactive application design?"
# query1 = "What is software maintenance?"
# query1 = "What is interaction design?"

In [13]:
answer = docsearch.similarity_search(query1)

In [14]:
print(answer[0].page_content)

REENGINEERING 9
Each of the aforementioned activities is made up of tasks described with specific
inputs, outputs, and actions.
SoftwareConfigurationManagementConfigurationmanagement(CM)isthedis-
cipline of managing changes in large systems. The goal of CM is to manage and
control the various extensions, adaptations, and corrections that are applied to a sys-
tem over its lifetime. It handles the control of all products/configuration items and
changes to those items. Software configuration management (SCM) is the config-
uration management applied to software systems. SCM is the means by which the
process of software evolution is managed. SCM has been defined in the IEEE 1042
standard[45]as“softwareconfigurationmanagement(SCM)isthedisciplineofman-
aging and controlling change in the evolution of software systems.” SCM provides a
framework for managing changes in a controlled manner. The purpose of SCM is to
reduce communication errors among personnel working on different aspects of the

In [15]:
answer_score = docsearch.similarity_search_with_score(query1) # closer to 0 is better

In [16]:
answer_score

[(Document(metadata={'source': 'data\\software-evolution-and-maintenance.pdf', 'page': 32}, page_content='REENGINEERING 9\nEach of the aforementioned activities is made up of tasks described with specific\ninputs, outputs, and actions.\nSoftwareConfigurationManagementConfigurationmanagement(CM)isthedis-\ncipline of managing changes in large systems. The goal of CM is to manage and\ncontrol the various extensions, adaptations, and corrections that are applied to a sys-\ntem over its lifetime. It handles the control of all products/configuration items and\nchanges to those items. Software configuration management (SCM) is the config-\nuration management applied to software systems. SCM is the means by which the\nprocess of software evolution is managed. SCM has been defined in the IEEE 1042\nstandard[45]as“softwareconfigurationmanagement(SCM)isthedisciplineofman-\naging and controlling change in the evolution of software systems.” SCM provides a\nframework for managing changes in a contr

### import chat

In [17]:
from langchain.chains.question_answering import load_qa_chain
from langchain.chains import RetrievalQA

### HuggingFace

In [18]:
from dotenv import load_dotenv
import os
from huggingface_hub import login

# Load environment variables
load_dotenv()
hf_token = os.getenv("HUGGINGFACE_API_KEY")

if not hf_token:
    raise ValueError("HUGGINGFACE_API_KEY not found in environment variables")

# Set the environment variable that HuggingFaceHub looks for
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

# Login to HuggingFace
login(token=hf_token)

# Initialize the model with the token explicitly passed
llm = HuggingFaceHub(
    repo_id="google/flan-t5-large",
    model_kwargs={"temperature": 0, "max_length": 512},
    huggingfacehub_api_token=hf_token  # Explicitly pass the token here
)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_20456\4225697184.py:19: LangChainDeprecationWarning: The class `HuggingFaceHub` was deprecated in LangChain 0.0.21 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEndpoint``.
  llm = HuggingFaceHub(


### get answers from your own docs 

In [19]:
# helper function to process the response from the QA chain
# and isloate result and source docs and page number

def parse_response(response):
    print(response['result'])
    print('\n\nSource:')
    for source_name in response['source_documents']:
        print(source_name.metadata['source'], "page #:", source_name.metadata['page'])

In [20]:
# setup the retriever on the faiss vector store
# make sure to set include_metadata = True

retriever = docsearch.as_retriever(include_metadata=True, metadata_key='source')

In [21]:
# setup the RetrieverQA chain with the retriever 
# make sure to return_source_documents = True

qa_chain = RetrievalQA.from_chain_type(llm=llm,
                                       chain_type="stuff",
                                       retriever=retriever,
                                       return_source_documents=True)

In [22]:
query = "Explain the three areas of the SCM (Software ConfigurationManagement) functionalities."
# query = "what is seven-segment display."
# query = "Who is the president of the United States?"
# query = "List down design values in interaction designs on interactive application design."
# query = "Why Personas Are Effective in interactive application design?"
# query = "What is software maintenance?"
# query = "Explain the characteristic of the personas."
# query = "What does that mean, 'continuing change' in the laws of Lehman?"
# query = "What are the two methods use to resolve version conflicts?"
# query = "What is interaction design?"
# query = "What is meant by Software Aging and code decay?"
# query = "What are the two types of prototyping in interaction design?"
# query = "what are the Three-Schema Architecture of database systems"
# query = "what are edges and pulses in the seven-segment display?"
# query = "Three different types of postures: in the context of interaction design process."
# query = "Discuss four types of information systems used in an organization."
# query = "Seven principles that focus on software engineering practice as a whole"
# query = "The CMMI represents a process meta model in two different ways: what are they?"
# query = "What are the Process Pattern Types"
# query = "The RUP (Rational Unified Process is normally described from three perspectives. What are they?"
# query = "State three Principles to Support Usability."

In [23]:
response = qa_chain(query)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_20456\1509190110.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = qa_chain(query)


In [24]:
type(response)

dict

In [25]:
response

{'query': 'Explain the three areas of the SCM (Software ConfigurationManagement) functionalities.',
 'result': 'SCM provides a framework for managing changes in a controlled manner. The purpose of SCM is to reduce communication errors among personnel working on different aspects of the softwareprojectbyprovidingacentralrepositoryofinformationabouttheprojectand a set of agreed upon procedures for coping with changes. It ensures that the released software is not contaminated by uncontrolled or unapproved changes.',
 'source_documents': [Document(metadata={'source': 'data\\software-evolution-and-maintenance.pdf', 'page': 32}, page_content='REENGINEERING 9\nEach of the aforementioned activities is made up of tasks described with specific\ninputs, outputs, and actions.\nSoftwareConfigurationManagementConfigurationmanagement(CM)isthedis-\ncipline of managing changes in large systems. The goal of CM is to manage and\ncontrol the various extensions, adaptations, and corrections that are applie

In [26]:
parse_response(response)

SCM provides a framework for managing changes in a controlled manner. The purpose of SCM is to reduce communication errors among personnel working on different aspects of the softwareprojectbyprovidingacentralrepositoryofinformationabouttheprojectand a set of agreed upon procedures for coping with changes. It ensures that the released software is not contaminated by uncontrolled or unapproved changes.


Source:
data\software-evolution-and-maintenance.pdf page #: 32
data\software-evolution-and-maintenance.pdf page #: 134
data\software-evolution-and-maintenance.pdf page #: 32
data\software-evolution-and-maintenance.pdf page #: 140


In [27]:
response['source_documents']

[Document(metadata={'source': 'data\\software-evolution-and-maintenance.pdf', 'page': 32}, page_content='REENGINEERING 9\nEach of the aforementioned activities is made up of tasks described with specific\ninputs, outputs, and actions.\nSoftwareConfigurationManagementConfigurationmanagement(CM)isthedis-\ncipline of managing changes in large systems. The goal of CM is to manage and\ncontrol the various extensions, adaptations, and corrections that are applied to a sys-\ntem over its lifetime. It handles the control of all products/configuration items and\nchanges to those items. Software configuration management (SCM) is the config-\nuration management applied to software systems. SCM is the means by which the\nprocess of software evolution is managed. SCM has been defined in the IEEE 1042\nstandard[45]as“softwareconfigurationmanagement(SCM)isthedisciplineofman-\naging and controlling change in the evolution of software systems.” SCM provides a\nframework for managing changes in a contro

### use Vector agent

In [35]:
# import the dependencies

from langchain.agents.agent_toolkits import (
    create_vectorstore_agent,
    VectorStoreToolkit,
    VectorStoreInfo
)

In [36]:
# set up the vectorstore info

vectorstore_info = VectorStoreInfo(
    name="pdf_vectorstore",
    description="pdf vectorstore",
    vectorstore=docsearch,
)

In [37]:
# Setup the VectorStoreToolkit and VectorStoreAgent

toolkit = VectorStoreToolkit(llm=llm, vectorstore_info=vectorstore_info)
agent_executor = create_vectorstore_agent(llm=llm,
                                          toolkit=toolkit,
                                          verbose=False)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_19652\1671463676.py:4: LangChainDeprecationWarning: See API reference for this function for a replacement implementation: https://api.python.langchain.com/en/latest/agents/langchain.agents.agent_toolkits.vectorstore.base.create_vectorstore_agent.html Read more here on how to create agents that query vector stores: https://python.langchain.com/docs/how_to/qa_chat_history_how_to/#agents
  agent_executor = create_vectorstore_agent(llm=llm,


In [38]:
# Add the string to ask for source

query = query + " List the sources."
print(query)

Explain the three areas of the SCM (Software ConfigurationManagement) functionalities. List the sources.


In [39]:
# run the agent

response = agent_executor.run(query)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_19652\4200706017.py:3: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = agent_executor.run(query)


ValueError: An output parsing error occurred. In order to pass this error back to the agent and have it try again, pass `handle_parsing_errors=True` to the AgentExecutor. This is the error: Could not parse LLM output: `I don't know`
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE

In [40]:
type(response)

dict

In [41]:
response

{'query': 'Explain the three areas of the SCM (Software ConfigurationManagement) functionalities.',
 'result': 'SCM provides a framework for managing changes in a controlled manner. The purpose of SCM is to reduce communication errors among personnel working on different aspects of the softwareprojectbyprovidingacentralrepositoryofinformationabouttheprojectand a set of agreed upon procedures for coping with changes. It ensures that the released software is not contaminated by uncontrolled or unapproved changes.',
 'source_documents': [Document(metadata={'source': 'data\\software-evolution-and-maintenance.pdf', 'page': 32}, page_content='REENGINEERING 9\nEach of the aforementioned activities is made up of tasks described with specific\ninputs, outputs, and actions.\nSoftwareConfigurationManagementConfigurationmanagement(CM)isthedis-\ncipline of managing changes in large systems. The goal of CM is to manage and\ncontrol the various extensions, adaptations, and corrections that are applie

RAGAs Framework for Ecaluations

In [28]:
pip install ragas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Evaluation for HuggingFace

In [29]:
pip install langchain-community huggingface-hub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
pip install huggingface_hub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

# Load environment variables
load_dotenv()

# Login to HuggingFace
hf_token = os.getenv("HUGGINGFACE_API_KEY")
if not hf_token:
    raise ValueError("HUGGINGFACE_API_KEY not found in environment variables")
login(token=hf_token)

In [32]:
# Install required package
!pip install sentence-transformers


[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from ragas import evaluate
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall
)
from datasets import Dataset
from langchain_community.llms import HuggingFaceHub
from langchain_community.embeddings import HuggingFaceEmbeddings
import time

# Initialize HuggingFace LLM with a model better suited for structured output
llm = HuggingFaceHub(
    repo_id="tiiuae/falcon-7b-instruct",  # Using a more capable model
    huggingfacehub_api_token=hf_token,
    model_kwargs={
        "temperature": 0.1,
        "max_new_tokens": 256,
        "do_sample": False
    }
)

# Initialize embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

# Create dataset with limited context
dataset = Dataset.from_dict({
    'question': ["Explain the three areas of the SCM (Software Configuration Management) functionalities."],
    'contexts': [[doc.page_content[:500] for doc in response['source_documents']]],
    'response': [qa_chain.invoke({"query": "Explain the three areas of the SCM (Software Configuration Management) functionalities."})['result']],
    'reference': ["The three areas of SCM functionalities are product, tool, and process. Product area involves identifying items to be managed, tool area involves using SCM systems to manage item evolution, and process area involves planning and configuration control."]
})

# Define metrics without custom parameters
metrics = [
    Faithfulness(),
    AnswerRelevancy(),
    ContextPrecision(),
    ContextRecall()
]

def evaluate_with_retry(dataset, metric, max_retries=3, delay=10):
    for attempt in range(max_retries):
        try:
            result = evaluate(
                dataset=dataset,
                metrics=[metric],
                llm=llm,
                embeddings=embeddings,
                raise_exceptions=True
            )
            return result
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"Failed after {max_retries} attempts: {str(e)}")
                return None
            print(f"Attempt {attempt + 1} failed, retrying in {delay} seconds...")
            time.sleep(delay)

print("\nEvaluating metrics individually:")
print("--------------------------------")

for metric in metrics:
    print(f"\nEvaluating {metric.__class__.__name__}...")
    result = evaluate_with_retry(dataset, metric)
    
    if result is not None:
        metric_name = metric.__class__.__name__.lower()
        if hasattr(result, metric_name):
            score = getattr(result, metric_name)
            print(f"{metric.__class__.__name__}: {score:.3f}")
        else:
            # Try to extract score from dictionary format
            if isinstance(result, dict) and metric_name in result:
                print(f"{metric.__class__.__name__}: {result[metric_name]:.3f}")
            else:
                print(f"{metric.__class__.__name__}: Score not available")
                print(f"Raw result: {result}")
    else:
        print(f"{metric.__class__.__name__}: Evaluation failed")

NameError: name 'hf_token' is not defined

In [34]:
from datasets import Dataset
from langchain_community.llms import HuggingFaceHub
from langchain_community.embeddings import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import logging
from typing import List, Dict
import time

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class QAEvaluator:
    def __init__(self, qa_chain, hf_token: str):
        self.qa_chain = qa_chain
        self.hf_token = hf_token
        self.encoder = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
        
    def calculate_similarity(self, text1: str, text2: str) -> float:
        """Calculate cosine similarity between two texts"""
        emb1 = self.encoder.encode([text1])
        emb2 = self.encoder.encode([text2])
        return float(cosine_similarity(emb1, emb2)[0][0])
    
    def evaluate_answer(self, question: str, answer: str, reference: str, context: str) -> Dict[str, float]:
        """Evaluate a single QA pair"""
        scores = {}
        
        # Answer Relevancy (similarity between answer and reference)
        scores['answer_relevancy'] = self.calculate_similarity(answer, reference)
        
        # Context Relevancy (similarity between answer and context)
        scores['context_relevancy'] = self.calculate_similarity(answer, context)
        
        # Question Relevancy (similarity between question and answer)
        scores['question_relevancy'] = self.calculate_similarity(question, answer)
        
        # Overall score (average of all metrics)
        scores['overall_score'] = np.mean(list(scores.values()))
        
        return scores
    
    def evaluate_qa_system(self) -> Dict[str, float]:
        """Evaluate QA system with test cases"""
        test_cases = [
            {
                "question": "What are the three SCM areas?",
                "reference": "SCM has three areas: product, tool, and process.",
            },
            {
                "question": "Define software maintenance.",
                "reference": "Software maintenance is modifying software after delivery to fix bugs and improve performance.",
            }
        ]
        
        all_scores = []
        try:
            for case in test_cases:
                logger.info(f"\nEvaluating question: {case['question']}")
                
                # Get answer from QA chain
                result = self.qa_chain.invoke({"query": case["question"]})
                answer = result['result']
                context = ' '.join([doc.page_content[:256] for doc in result['source_documents'][:2]])
                
                # Evaluate answer
                scores = self.evaluate_answer(
                    case["question"],
                    answer,
                    case["reference"],
                    context
                )
                
                logger.info(f"Answer: {answer[:100]}...")
                logger.info("Scores:")
                for metric, score in scores.items():
                    logger.info(f"{metric}: {score:.3f}")
                
                all_scores.append(scores)
            
            # Calculate average scores across all test cases
            avg_scores = {}
            for metric in all_scores[0].keys():
                avg_scores[metric] = np.mean([s[metric] for s in all_scores])
            
            return avg_scores
                    
        except Exception as e:
            logger.error(f"Evaluation failed: {str(e)}")
            return {}

# Main execution
if __name__ == "__main__":
    try:
        print("\nStarting QA System Evaluation")
        print("----------------------------")
        
        if not hf_token:
            raise ValueError("HuggingFace API token not found!")
        
        evaluator = QAEvaluator(qa_chain, hf_token)
        results = evaluator.evaluate_qa_system()
        
        if results:
            print("\nEvaluation Results:")
            print("------------------")
            for metric, score in results.items():
                print(f"{metric}: {score:.3f}")
        else:
            print("\nNo evaluation results were returned.")
            
    except Exception as e:
        print(f"\nEvaluation failed with error: {str(e)}")
        logger.exception("Detailed error information:")

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-mpnet-base-v2



Starting QA System Evaluation
----------------------------


INFO:__main__:
Evaluating question: What are the three SCM areas?
Batches: 100%|██████████| 1/1 [00:00<00:00, 15.90it/s]
INFO:__main__:Answer: SCM functionalities Identification Version control System models and selection Workspace control Bui...
INFO:__main__:Scores:
INFO:__main__:answer_relevancy: 0.634
INFO:__main__:context_relevancy: 0.787
INFO:__main__:question_relevancy: 0.487
INFO:__main__:overall_score: 0.636
INFO:__main__:
Evaluating question: Define software maintenance.
Batches: 100%|██████████| 1/1 [00:00<00:00, 10.42it/s]
INFO:__main__:Answer: The ISO/IEC 14764 standard [3] defines software maintenance as “...the totality ofactivitiesrequired...
INFO:__main__:Scores:
INFO:__main__:answer_relevancy: 0.716
INFO:__main__:context_relevancy: 0.731
INFO:__main__:question_relevancy: 0.812
INFO:__main__:overall_score: 0.753



Evaluation Results:
------------------
answer_relevancy: 0.675
context_relevancy: 0.759
question_relevancy: 0.650
overall_score: 0.695
